In [20]:


import re
import datetime
import pandas as pd

df = pd.read_csv("dataset_sintetico_regex_ruidos.csv")

df.head()

,nome,email,telefone,cpf,cnpj,cep,data,valor,url,texto_livre
0,Danielë Souza,o ig8f_1c@example.com,(71) 959440-7816,999.9299.9999-99,86.379.402/6542-35,84959-310,331/07/203,"R$5 3.553,18",http://site.org/3xkxw/rek8,Cliente: Danielë Souza Contato: o ig8f_1c@exam...
1,S érgi8 Rodrigues,87a+u5b@,+55 71 57871ê331,43039117b122,2 27824-9638308,09839301,33/6/1994,141.556,h ttps://exemplo.com/n8i4p4/mgg1w11/3dgdzv,Cliente: S érgi8 Rodrigues Contato: 87a+u5b@ |...
2,H elena Costa,3uea3-.geû@ufc.br,3191824-4935,5555555555,54278498084187,48740164,13//11/2005,"_1,371.3",http://sitee.org/919ah,Cliente: H elena Costa Contato: 3uea3-.geû@ufc...
3,Yasmin Martins,c1a7mx1e@gmail com,(98) 95777-3872,951.484.656-70,36.629.946/8044-38,4895343,3 9/088/2029,"R $ 3.497,38",http://site.orrg/my5zpj/g1o,Cliente: Yasmin Martins Contato: c1a7mx1e@gmai...
4,Natan Azevedo,-8bz_.b@example.com,+55(92)90330-9232,272.0946.537-26,64641708053143,1 9374-5229,1 9/02/2005,R$ 777.92,https://contato.me/bty/mepgth,Cliente: Natan Azevedo Contato: -8bz_.b@exampl...


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   nome         1000 non-null   object
 1   email        1000 non-null   object
 2   telefone     1000 non-null   object
 3   cpf          1000 non-null   object
 4   cnpj         1000 non-null   object
 5   cep          1000 non-null   object
 6   data         1000 non-null   object
 7   valor        1000 non-null   object
 8   url          1000 non-null   object
 9   texto_livre  1000 non-null   object
dtypes: object(10)
memory usage: 78.3+ KB


In [22]:
rx_email = re.compile(r"""
    ^[A-Za-z0-9._%+-]+      # parte local
    @
    [A-Za-z0-9.-]+          # domínio
    \.[A-Za-z]{2,}$         # TLD
""", re.X)

RX_CPF = re.compile(r"(?<!\d)(?:\d{3}\.?\d{3}\.?\d{3}-?\d{2})(?!\d)")
RX_CNPJ = re.compile(r"(?<!\d)(?:\d{2}\.?\d{3}\.?\d{3}/?\d{4}-?\d{2})(?!\d)")
RX_CEP = re.compile(r"(?<!\d)\d{5}-?\d{3}(?!\d)")
RX_DATA = re.compile(r"(?<!\d)(0?[1-9]|[12]\d|3[01])/(0?[1-9]|1[0-2])/\d{4}(?!\d)")
RX_FONE = re.compile(r"(?x)(?:\+?55\s*)?(?:\(?\d{2}\)?\s*)?(?:9?\d{4})-?\s?\d{4}")
RX_EMAIL = rx_email
RX_URL = re.compile(r"https?://[^\s/$.?#].[^\s]*", re.I)
RX_NOME = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ]+(?:\s+[A-Za-zÀ-ÖØ-öø-ÿ]+)+")
RX_TELEFONE = re.compile(
    r"(?:\+?55\s*)?(?:\(?\d{2}\)?\s*)?\d{4,5}-?\d{4}"
)

In [23]:
texto = df.loc[0, "texto_livre"]

print("CPFs no texto:", RX_CPF.findall(texto))
print("E-mails no texto:", RX_EMAIL.findall(texto.replace(" ", "")))
print("Telefones no texto:", RX_TELEFONE.findall(texto))

CPFs no texto: []
E-mails no texto: []
Telefones no texto: ['959440-7816']


In [24]:
def so_digitos(s: str) -> str:
    return re.sub(r"\D", "", s or "")


In [25]:
def normaliza_nome(texto: str) -> str | None:
    if not isinstance(texto, str):
        return None

    # 1) Mantém só letras (com acento) e espaços
    s = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿ\s]", " ", texto)
    s = re.sub(r"\s+", " ", s).strip()
    if not s:
        return None

    tokens = s.split()
    merged = []
    i = 0

    while i < len(tokens):
        t = tokens[i]

        # se for letra única e tiver próxima palavra maior, junta as duas
        if len(t) == 1 and i + 1 < len(tokens) and len(tokens[i+1]) > 1:
            merged.append(t + tokens[i+1])   # "H" + "elena" -> "Helena"
            i += 2
        elif len(t) == 1:
            # letra sozinha no fim ou seguida de outra letra sozinha → ignora
            i += 1
        else:
            merged.append(t)
            i += 1

    if not merged:
        return None

    return " ".join(merged).title()

df["nome_normalizado"] = df["nome"].apply(normaliza_nome)


In [26]:
def formata_cpf(d: str) -> str:
    return f"{d[:3]}.{d[3:6]}.{d[6:9]}-{d[9:]}"

def normaliza_cpf(cpf: str) -> str | None:
    d = so_digitos(cpf)
    if len(d) != 11:
        return None
    return formata_cpf(d)

df["cpf_normalizado"] = df["cpf"].apply(normaliza_cpf)


In [27]:
def formata_cnpj(d: str) -> str:
    return f"{d[:2]}.{d[2:5]}.{d[5:8]}/{d[8:12]}-{d[12:]}"

def normaliza_cnpj(cnpj: str) -> str | None:
    d = so_digitos(cnpj)
    if len(d) != 14:
        return None
    return formata_cnpj(d)

df["cnpj_normalizado"] = df["cnpj"].apply(normaliza_cnpj)


In [28]:
def normaliza_telefone(fone: str) -> str | None:
    d = so_digitos(fone)
    if not d:
        return None

    # Remove código do país 55 se vier
    if d.startswith("55") and len(d) > 11:
        d = d[2:]

    # Aceita 10 ou 11 dígitos (DDD + número)
    if len(d) == 11:
        ddd = d[:2]
        num = d[2:]
    elif len(d) == 10:
        ddd = d[:2]
        num = "9" + d[2:]  # insere o 9 no começo
    else:
        return None  # demais casos eu marco como inválido

    return f"({ddd}) {num[0]}{num[1:5]}-{num[5:]}"

df["telefone_normalizado"] = df["telefone"].apply(normaliza_telefone)


In [29]:
def normaliza_cep(cep: str) -> str | None:
    d = so_digitos(cep)
    if len(d) != 8:
        return None
    return f"{d[:5]}-{d[5:]}"

df["cep_normalizado"] = df["cep"].apply(normaliza_cep)


In [30]:
def normaliza_data(texto: str) -> str | None:
    if not isinstance(texto, str):
        return None

    # pega a primeira sequência dd qualquer / mm qualquer / aaaa (4 dígitos)
    m = re.search(r"(\d{1,2})\D+(\d{1,2})\D+(\d{4})", texto)
    if not m:
        return None

    d, mth, y = map(int, m.groups())

    # restringe ano para algo razoável
    if not (1900 <= y <= 2100):
        return None

    try:
        dt = datetime.date(y, mth, d)
    except ValueError:
        return None

    return dt.strftime("%d/%m/%Y")

df["data_normalizada"] = df["data"].apply(normaliza_data)


In [31]:
def extrai_valor_monetario(texto: str) -> float | None:
    if not isinstance(texto, str):
        return None

    # padrão brasileiro com milhar e vírgula: 1.234,56
    m = re.search(r"-?\d{1,3}(?:\.\d{3})*(?:,\d{2})", texto)
    if m:
        s = m.group(0).replace(".", "").replace(",", ".")
        try:
            return float(s)
        except ValueError:
            return None

    # fallback: 1234.56
    m = re.search(r"-?\d+\.\d{2}", texto)
    if m:
        try:
            return float(m.group(0))
        except ValueError:
            return None

    # último recurso: inteiro
    m = re.search(r"-?\d+", texto)
    return float(m.group(0)) if m else None

df["valor_float"] = df["valor"].apply(extrai_valor_monetario)


In [32]:
RX_EMAIL = re.compile(r"[A-Za-z0-9_.+-]+@[A-Za-z0-9-]+\.[A-Za-z0-9-.]+")
RX_URL   = re.compile(r"https?://[A-Za-z0-9./%-_]+")

def normaliza_email(texto: str) -> str | None:
    if not isinstance(texto, str):
        return None
    # remove espaços no meio, que quebram o e-mail
    m = RX_EMAIL.search(texto.replace(" ", ""))
    return m.group(0) if m else None

def normaliza_url(texto: str) -> str | None:
    if not isinstance(texto, str):
        return None
    m = RX_URL.search(texto.replace(" ", ""))
    return m.group(0) if m else None

df["email_normalizado"] = df["email"].apply(normaliza_email)
df["url_normalizada"]   = df["url"].apply(normaliza_url)


In [33]:
def valida_cpf(cpf: str) -> bool:
    d = so_digitos(cpf)
    if len(d) != 11:
        return False

    # descarta sequências tipo 11111111111
    if len(set(d)) == 1:
        return False

    # primeiro dígito verificador
    soma = sum(int(d[i]) * (10 - i) for i in range(9))
    resto = soma % 11
    dig1 = 0 if resto < 2 else 11 - resto
    if dig1 != int(d[9]):
        return False

    # segundo dígito verificador
    soma = sum(int(d[i]) * (11 - i) for i in range(10))
    resto = soma % 11
    dig2 = 0 if resto < 2 else 11 - resto
    if dig2 != int(d[10]):
        return False

    return True

df["cpf_valido"] = df["cpf"].apply(valida_cpf)


In [34]:
def valida_cnpj(cnpj: str) -> bool:
    d = so_digitos(cnpj)
    if len(d) != 14:
        return False

    if len(set(d)) == 1:
        return False

    pesos1 = [5,4,3,2,9,8,7,6,5,4,3,2]
    soma = sum(int(d[i]) * pesos1[i] for i in range(12))
    resto = soma % 11
    dig1 = 0 if resto < 2 else 11 - resto
    if dig1 != int(d[12]):
        return False

    pesos2 = [6] + pesos1
    soma = sum(int(d[i]) * pesos2[i] for i in range(13))
    resto = soma % 11
    dig2 = 0 if resto < 2 else 11 - resto
    if dig2 != int(d[13]):
        return False

    return True

df["cnpj_valido"] = df["cnpj"].apply(valida_cnpj)


In [35]:
n_total = len(df)

resumo = {
    "cpf_validos": int(df["cpf_valido"].sum()),
    "cpf_invalidos": int((~df["cpf_valido"]).sum()),
    "cnpj_validos": int(df["cnpj_valido"].sum()),
    "cnpj_invalidos": int((~df["cnpj_valido"]).sum()),
    "telefones_normalizados": int(df["telefone_normalizado"].notna().sum()),
    "ceps_normalizados": int(df["cep_normalizado"].notna().sum()),
    "datas_normalizadas": int(df["data_normalizada"].notna().sum()),
    "emails_normalizados": int(df["email_normalizado"].notna().sum()),
    "urls_normalizadas": int(df["url_normalizada"].notna().sum()),
    "valores_preenchidos": int(df["valor_float"].notna().sum()),
}

resumo, n_total


({'cpf_validos': 465,
  'cpf_invalidos': 535,
  'cnpj_validos': 475,
  'cnpj_invalidos': 525,
  'telefones_normalizados': 817,
  'ceps_normalizados': 619,
  'datas_normalizadas': 594,
  'emails_normalizados': 757,
  'urls_normalizadas': 848,
  'valores_preenchidos': 1000},
 1000)

In [37]:
df.to_csv("dataset_sintetico_regex_limpo.csv", index=False, encoding="utf-8")